# HTx 2024 IMPACT Model

## Setup and Imports

In [ ]:
%load_ext autoreload
%autoreload 2

# Set CPU fallback for MPS (Apple Silicon)
import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
print("PYTORCH_ENABLE_MPS_FALLBACK =", os.environ.get('PYTORCH_ENABLE_MPS_FALLBACK'))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import logging

from datetime import datetime
from prism.config import INTERIM_DATA_DIR, MODELS_DIR, PROCESSED_DATA_DIR
from prism.config import DATASET_PREFIX, CONFIG_NAME
from prism.logging_config import setup_logging
from prism.config_loader import load_config_if_exists

# =============================================================================
# DATASET CHECK - Ensure .env is configured
# =============================================================================
if DATASET_PREFIX is None:
    print("=" * 70)
    print("ERROR: No dataset configured!")
    print("=" * 70)
    print("\nPlease configure your dataset in the .env file:")
    print("  1. Copy .env.example to .env")
    print("  2. Set PRISM_CONFIG=your_config (loads example_notebooks/config/your_config.yaml)")
    print("     OR set PRISM_DATASET=your_dataset (quick mode, no YAML needed)")
    print("\nExample .env contents:")
    print("  PRISM_CONFIG=htx_example")
    print("=" * 70)
    raise ValueError("DATASET_PREFIX is None - configure .env file")

print(f"Dataset: {DATASET_PREFIX}")
if CONFIG_NAME:
    print(f"Config: {CONFIG_NAME}.yaml")

from prism.device_tools import get_device, get_num_cpu_workers
from prism.data_utilities import feature_summary, save_split_datasets
from prism.metrics import evaluate_model_performance
from prism.impact import apply_impact_model_torch, IMPACTModel
from prism.preprocessing import PRiSMScaler

In [ ]:
model_name = 'impact'

In [ ]:
# Check the data prefix that will be used to load train, test, and validation cohorts
# This prefix is used to construct filenames like "{DATASET_PREFIX}_train.csv" from INTERIM_DATA_DIR
print(f"DATASET_PREFIX = '{DATASET_PREFIX}'")
print(f"Will load data files from {INTERIM_DATA_DIR}:")

# Define filename variables (can be manually overridden if needed)
train_filename = f'{DATASET_PREFIX}_train.csv'
test_filename = f'{DATASET_PREFIX}_test.csv'
val_filename = f'{DATASET_PREFIX}_val.csv'

print(f"  - {train_filename}")
print(f"  - {test_filename}") 
print(f"  - {val_filename}")

In [ ]:
model_identifier = f"{DATASET_PREFIX}_{model_name}"
print(model_identifier)

In [ ]:
# Set up logging (use logging.DEBUG for more detailed logs, logging.INFO for less detailed logs)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
setup_logging(
    file_log_level=logging.INFO,
    log_file=f'model_{model_identifier}_{timestamp}.log',
    console_output=True,
    root_log_level=logging.WARNING,
)

In [ ]:
# Set device to 'cpu', or GPU ('cuda' for NVIDIA, 'mps' for Apple Silicon)
# By default, we select the best available device
device = get_device() 
print(f"Using device: {device}")

# Set random seed for reproducibility
random_seed = 257
np.random.seed(random_seed)
torch.manual_seed(random_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(random_seed)
    torch.cuda.manual_seed_all(random_seed)

# Force deterministic algorithms (may reduce performance)
# torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark = False
# torch.use_deterministic_algorithms(True)

# Set saving parameters
SAVE_DATA = True
SAVE_MODELS = True # Save instance of IMPACT model for compatability with load notebook

In [ ]:
# Set number of workers for multithreading
max_workers = get_num_cpu_workers()
print(f"max_workers = {max_workers}")

## Load Data and Modify Features

In [ ]:
# Load data
data_train = pd.read_csv(INTERIM_DATA_DIR.joinpath(train_filename))
data_test = pd.read_csv(INTERIM_DATA_DIR.joinpath(test_filename))
data_val = pd.read_csv(INTERIM_DATA_DIR.joinpath(val_filename))

In [ ]:
data_test

In [ ]:
# ============================================================================
# TARGET AND ID VARIABLE CONFIGURATION
# ============================================================================
# IMPACT model defaults - specific to heart transplant data
# These can be overridden by config file or modified manually below:
target_column = 'event_oneyear'
id_column = 'trr_id_code'

# Load config file to optionally override target/ID
config = load_config_if_exists(DATASET_PREFIX)

if config:
    print(f"Config file found: {DATASET_PREFIX}.yaml")
    if 'target_candidates' in config and config['target_candidates']:
        # Use first target candidate if provided
        target_column = config['target_candidates'][0]
        print(f"   [ok] target_column: {target_column}")
    if 'id_candidates' in config and config['id_candidates']:
        # Use first ID candidate if provided
        id_column = config['id_candidates'][0]
        print(f"   [ok] id_column: {id_column}")
else:
    print("No config file found - using IMPACT model defaults")
    print(f"  target_column: {target_column}")
    print(f"  id_column: {id_column}")

# Note: Reference columns have already been dropped during preprocessing
# (diagn_Cardiomyopathy and recethcat_Caucasian were dropped with drop_reference_columns=True)
print("[INFO] Reference columns already dropped during preprocessing")
print("  diagn_Cardiomyopathy (reference for diagn group)")
print("  recethcat_Caucasian (reference for recethcat group)")

drop_cols = [target_column, id_column]

No changes have been made to the dataset, but for competeness we save it to `PROCESSED_DATA_DIR`.

In [ ]:
if SAVE_DATA:  
    # Save the modified data used for modelling
    # Note: Reference columns have already been dropped during preprocessing
    print("Saving processed data (reference columns already dropped)...")
    
    save_split_datasets(
        train_df=data_train,
        test_df=data_test,
        val_df=data_val,
        base_filename=model_identifier,
        save_dir=PROCESSED_DATA_DIR,
        header_comment=f"target: {target_column}, patient IDs: {id_column}. All other features are model inputs, pre-normalization. Reference columns were dropped during preprocessing.",
    )

In [ ]:
X_train = data_train.drop(drop_cols, axis=1)
y_train = data_train[target_column]

X_test = data_test.drop(drop_cols, axis=1)
y_test = data_test[target_column]

X_val = data_val.drop(drop_cols, axis=1)
y_val = data_val[target_column]

In [ ]:
# Check class distribution
print("\nClass distribution in training set:")
print(y_train.value_counts(normalize=True))
print("\nClass distribution in testing set:")
print(y_test.value_counts(normalize=True))

In [ ]:
# Get feature names
feature_names = X_train.columns.tolist()

## Data Overview

In [ ]:
# Generate feature statistics
feat_summary = feature_summary(X_train, categorical_threshold=15)
print("Feature Statistics:")
display(feat_summary)

In [ ]:
# Validate preprocessing requirements for IMPACT model
print("\nValidating IMPACT model input requirements:")

# Check bilirubin units (should be in umol/L)
bil_min, bil_max = X_train['recbilirubin'].min(), X_train['recbilirubin'].max()
print(f"Bilirubin range: {bil_min:.1f} - {bil_max:.1f} umol/L")
# Normal range for umol/L: ~5-150; if in mg/dL would be 0.1-8
if bil_min < 5 or bil_max > 200:
    print(f"  WARNING: Bilirubin range unusual for umol/L (expected ~5-150)")
if bil_max < 10:  # Strong indicator it's in mg/dL
    print(f"  ERROR: Bilirubin appears to be in mg/dL, not umol/L!")

# Check creatinine clearance (should be in mL/min)
creat_min, creat_max = X_train['creatinine_clearance'].min(), X_train['creatinine_clearance'].max()
print(f"Creatinine clearance range: {creat_min:.1f} - {creat_max:.1f} mL/min")
if creat_min < 5 or creat_max > 400:
    print(f"  WARNING: Creatinine clearance range unusual (expected ~10-250 mL/min)")
if creat_max > 500 or creat_min > 100:  # Likely raw creatinine in umol/L
    print(f"  ERROR: Values suggest raw serum creatinine, not clearance!")

# Check all required IMPACT features are present
required_features = IMPACTModel.FEATURE_NAMES
missing_features = [f for f in required_features if f not in X_train.columns]
if missing_features:
    print(f"  ERROR: Missing required features: {missing_features}")
else:
    print(f"All {len(required_features)} required IMPACT features present")

## Define the IMPACT model

The following IMPACT survival model is based on

Weiss ES, Allen JG, Arnaoutakis GJ, George TJ, Russell SD, Shah AS, Conte JV. Creation of a quantitative recipient risk index for mortality prediction after cardiac transplantation (IMPACT). Ann Thorac Surg. 2011 Sep;92(3):914-21; discussion 921-2. doi: [10.1016/j.athoracsur.2011.04.030](https://doi.org/10.1016/j.athoracsur.2011.04.030). PMID: 21871277.

| Variable | Description | Assumed Unit/Encoding | Notes |
|----------|-------------|-----------------|-------|
| `recageyear` | Recipient age | Years | Age > 60 gets 3 points |
| `recsex` | Recipient sex | 1 = Female, 0 = Male | Females get 3 points |
| `recweightkg` | Recipient weight | Kilograms | Used for creatinine clearance calculation |
| `recinfections2weeks` | Infection within 2 weeks | 1 = Yes, 0 = No | Yes gets 3 points |
| `recbilirubin` | Serum bilirubin | umol/L | 17.1-34.2: 1 point, 34.2-68.4: 3 points, ≥68.4: 4 points |
| `reccreatininemostrecent` | Serum creatinine | umol/L | Used to calculate creatinine clearance |
| `recdialysis` | Dialysis between listing and transplant | 1 = Yes, 0 = No | Yes gets 4 points |
| `rececmo` | Temporary circulatory support | 1 = Yes, 0 = No | Yes gets 7 points |
| `recvad` | Ventricular assist device | 1 = Yes, 0 = No | Assumed all are older gen pulsatile (3 points) |
| `reciabp` | Intraaortic balloon pump | 1 = Yes, 0 = No | Yes gets 3 points |
| `recventilator` | Mechanical ventilation | 1 = Yes, 0 = No | Yes gets 5 points |
| `recethcat_*` | Ethnicity categories | 1 = Yes, 0 = No | African American gets 3 points |
| `diagn_CAD` | Ischemic cardiomyopathy | 1 = Yes, 0 = No | Yes gets 2 points |
| `diagn_Cardiomyopathy` | Idiopathic cardiomyopathy | 1 = Yes, 0 = No | Reference category (0 points) |
| `diagn_Congenital` | Congenital heart disease | 1 = Yes, 0 = No | Yes gets 5 points |
| `diagn_Valve/Graftfailure/Misc` | Other diagnoses | 1 = Yes, 0 = No | Yes gets 1 point if no other diagnosis |

Key Assumptions:

1. Creatinine clearance is calculated using the Cockcroft-Gault formula with creatinine in umol/L

$$
\text{CrCl (mL/min)} = \frac{[140-\text{age (years)}] \times \text{weight (kg)}}{\text{plasma creatinine (umol/L)}} \times (1.04 \text{ if female else } 1.23 \text{ if male })
$$

2. All VADs are treated as older generation pulsatile devices (3 points) due to lack of detailed type information
3. Diagnostic categories are mapped as follows
  - Idiopathic → Cardiomyopathy
  - Ischemic → CAD (Coronary Artery Disease)
  - Congenital → Congenital
  - Other → Valve, Graft failure, Misc
4. Binary variables are all encoded as 1 for yes/presence and 0 for no/absence
5. Bilirubin is expected in umol/L. The original IMPACT paper (Weiss et al. 2011) used thresholds in mg/dL (1, 2, 4 mg/dL); the implementation uses pre-scaled thresholds (17.1, 34.2, 68.4 umol/L, where 1 mg/dL = 17.1 umol/L).

## Run the IMPACT Model On All Data

In [ ]:
data_train_impact = apply_impact_model_torch(data_train)
data_test_impact = apply_impact_model_torch(data_test)

## Evaluate Model Performance

In [ ]:
# Evaluate the model
y_pred_train_impact = data_train_impact['pred_impact']

results_impact_train = evaluate_model_performance(
    y_train,
    y_pred_train_impact,
    title=f"{model_identifier} - Training Data",
    threshold=0.5,
    random_seed=random_seed,
)

In [ ]:
y_pred_test_impact = data_test_impact['pred_impact']

results_impact_test = evaluate_model_performance(
    y_test,
    y_pred_test_impact,
    title=f"{model_identifier} - Test Data",
    threshold=0.5,
    random_seed=random_seed,
)

## Save Model

In [ ]:
# Save an instance of the IMPACT model for use with load notebook

blackbox_model = IMPACTModel()
scaler = PRiSMScaler(scaler=None) # dummy scaler for IMPACT model

if SAVE_MODELS:

    # Create timestamp for unique filename
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Gather hyperparameters
    hyperparameters = {}

    # Set filepath
    model_dir = MODELS_DIR.joinpath(model_identifier)
    os.makedirs(model_dir, exist_ok=True)

    model_filename = f"{model_identifier}_model_{timestamp}.pt"
    model_path = model_dir.joinpath(model_filename)

    # Save the model in PyTorch standard format
    torch.save(
        {
            'scaler': scaler,
            'model': blackbox_model,  # Full model for convenience
            'model_state_dict': blackbox_model.state_dict(),
            'model_class': type(blackbox_model),
            'hyperparameters': hyperparameters,
            'feature_names': blackbox_model.FEATURE_NAMES,
            'train_metrics': {},
            'test_metrics': {},
        },
        model_path,
    )

    print(f"Model also saved in standard PyTorch format to {model_path}")